# CHOMP optimizer — tests & comparison with the WarmStartPlanner

CHOMP is a classical, learning-free trajectory optimizer. It minimizes collision + joint-limits +
smoothness via a covariant gradient update with fixed start/goal, and can be warm-started from a
model trajectory.

**Two independent weight sets** (see `chomp.py`):

* **Optimization hyperparameters** (`eps`, `w_coll`, `w_smooth`, ...) drive the descent. Defaults are
  tuned for *single-trajectory* CHOMP. The Bigboy *training* weights (wide `eps=0.8` + strong
  smoothness) are great for training a network but make single-trajectory CHOMP straighten wide
  detours back into obstacles — so they are deliberately **not** the optimizer defaults.
* **Evaluation weights** (`eval_*`) define the **comparison metric** and default to the **Bigboy
  training objective**, so model and CHOMP are judged on exactly the loss the model was trained on.
  `evaluate_trajectory` always uses these, plus weight-independent geometric feasibility
  (`collision_free`, `n_collision_pts`, `min_clearance`) using the sphere *surface* clearance —
  the cleanest comparison signal.

**Headline finding:** CHOMP from a straight-line init almost always gets stuck in a local minimum on
this problem class. The learned model is far faster and more often collision-free, and
**model → CHOMP refinement is the best of both**.

In [ ]:
# ---- Setup ----
import sys, os, time
import numpy as np
import matplotlib.pyplot as plt
import torch

SIMPLEARM_PATH = os.path.abspath("../external/SimpleArm/src")
sys.path.insert(0, SIMPLEARM_PATH)

from simplearm.robot import RobotInfo
from simplearm.geom import Obstacles
from simplearm.viz import RobotViewer

from chomp import CHOMPOptimizer
import models
from visualization import save_viewer_as_gif

DEVICE = "cpu"
GRID_LENGTH = 2.5
torch.manual_seed(0)
np.random.seed(0)

## 1. Load the Bigboy dataset & trained model

`CHOMPOptimizer.from_metadata` builds robot / joint-limits / grid straight from the dataset metadata
(same setup as `training.train`).

In [ ]:
DATASET_PATH = "data/good_dataset_test.pt"
MODEL_PATH   = "models/warm_start_planner.pt"

ds   = torch.load(DATASET_PATH, weights_only=False)
meta = ds["metadata"]

chomp = CHOMPOptimizer.from_metadata(meta, T=50, device=DEVICE)
print("optimizer HP :", dict(eps=chomp.eps, w_coll=chomp.w_coll, w_smooth=chomp.w_smooth,
                             agg=chomp.collision_agg, eta=chomp.eta))
print("eval metric  :", dict(eps=chomp.eval_eps, w_coll=chomp.eval_w_coll,
                             w_smooth=chomp.eval_w_smooth, agg=chomp.eval_collision_agg))

robot = chomp.robot
model = models.WarmStartPlanner(dof=meta["dof"], T=50, C=10, linklengths=meta["linklengths"]).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()
print("\nLoaded model:", MODEL_PATH, "| dataset N =", meta["N"])

## 2. One sample: model vs. CHOMP-from-scratch vs. model→CHOMP

Pick a sample where the model output still collides. We expect: CHOMP-from-scratch reduces
collisions a lot but stays stuck; model→CHOMP clears it.

In [ ]:
# Find a sample where the model output collides but model->CHOMP fixes it
# (the compelling before/after). Fall back to the first colliding sample.
i, fallback = None, None
for k in range(min(40, meta["N"])):
    sk, qk, gk = ds["sdf"][k:k+1], ds["q_start"][k:k+1], ds["q_goal"][k:k+1]
    with torch.no_grad():
        wp_k = model(qk, gk, sk)
    if chomp.evaluate_trajectory(model.trajectory(wp_k), sk)["collision_free"]:
        continue
    if fallback is None:
        fallback = k
    tr_k = chomp.optimize(sk, qk, gk, init_waypoints=wp_k, max_iters=300)
    if chomp.evaluate_trajectory(tr_k, sk)["collision_free"]:
        i = k; break
i = i if i is not None else (fallback if fallback is not None else 0)
print(f"Using sample {i} (model output collides)")

sdf_i = ds["sdf"][i]
qs_i  = ds["q_start"][i:i+1]
qg_i  = ds["q_goal"][i:i+1]
n_obs = ds["n_obstacles"][i].item()
obs_i = ds["obstacles"][i, :n_obs]
obstacles_i = Obstacles(x=obs_i[:, 0].numpy(), y=obs_i[:, 1].numpy(), r=obs_i[:, 2].numpy())

with torch.no_grad():
    wp_i       = model(qs_i, qg_i, sdf_i.unsqueeze(0))
    traj_model = model.trajectory(wp_i)

# Optimize until convergence (collision-free OR cost plateau), capped at max_iters.
traj_scratch, hist = chomp.optimize(sdf_i, qs_i, qg_i, max_iters=500, return_history=True)
traj_refined       = chomp.optimize(sdf_i, qs_i, qg_i, init_waypoints=wp_i, max_iters=500)
print(f"CHOMP-from-scratch ran {len(hist)} iterations until convergence")

for name, tr in [("model", traj_model), ("CHOMP scratch", traj_scratch), ("model+CHOMP", traj_refined)]:
    e = chomp.evaluate_trajectory(tr, sdf_i)
    print(f"{name:14s} free={e['collision_free']!s:5s} collisions={e['n_collision_pts']:3d} "
          f"min_clearance={e['min_clearance']:+.3f}  total={e['total']:.2f}")

In [ ]:
# ---- CHOMP-from-scratch optimization cost vs. iteration ----
fig, ax = plt.subplots(figsize=(8, 3))
for key in ["total", "coll", "smooth"]:
    ax.plot([h[key] for h in hist], label=key)
ax.set_xlabel("Iteration"); ax.set_ylabel("Optimizer cost"); ax.set_yscale("log")
ax.legend(); ax.grid(True, which="both", alpha=0.3)
ax.set_title(f"CHOMP optimization cost vs. iteration (sample {i})")
plt.tight_layout(); plt.show()

In [ ]:
# ---- Before / after visualization (model vs. model+CHOMP) ----
for label, tr in [("model", traj_model), ("model + CHOMP", traj_refined)]:
    print(label)
    viz = RobotViewer(tr.squeeze(0).cpu().numpy(), robot, obstacles=obstacles_i, animate=True)
    viz.plot()

# Save the refined trajectory as a GIF
viz = RobotViewer(traj_refined.squeeze(0).cpu().numpy(), robot, obstacles=obstacles_i, animate=True)
viz.plot()
save_viewer_as_gif(viz, f"resources/model_chomp_sample_{i}.gif", duration_ms=60)

## 3. Speed & final-result comparison over many samples

The main comparison: collision-free rate, **iterations-to-convergence**, and wall-clock speed for
**model**, **CHOMP from scratch**, and **model→CHOMP refinement**, all judged on the same Bigboy
evaluation metric. CHOMP runs until convergence (collision-free *or* cost plateau, capped at
`max_iters`) — **not** a fixed iteration count — so the time/iteration numbers are meaningful. CHOMP
is still ~1000× slower per sample, so keep `N_COMPARE` modest.

In [ ]:
import pandas as pd

N_COMPARE = 20
idxs = list(range(N_COMPARE))
sdf  = ds["sdf"][idxs]
qs   = ds["q_start"][idxs]
qg   = ds["q_goal"][idxs]

# --- Model ---
t0 = time.time()
with torch.no_grad():
    wp = model(qs, qg, sdf)
    traj_model_b = model.trajectory(wp)
t_model = (time.time() - t0) / N_COMPARE
model_free = sum(chomp.evaluate_trajectory(traj_model_b[k:k+1], sdf[k:k+1])["collision_free"]
                 for k in range(N_COMPARE))


def run_chomp(use_warmstart):
    """Optimize each sample until convergence; return (free_count, ms/sample, mean_iters)."""
    free, iters, t0 = 0, [], time.time()
    for k in range(N_COMPARE):
        kw = dict(init_waypoints=wp[k:k+1]) if use_warmstart else {}
        tr, h = chomp.optimize(sdf[k:k+1], qs[k:k+1], qg[k:k+1], max_iters=500,
                               return_history=True, **kw)
        free += chomp.evaluate_trajectory(tr, sdf[k:k+1])["collision_free"]
        iters.append(len(h))
    return free, (time.time() - t0) / N_COMPARE, float(np.mean(iters))


scratch_free, t_scratch, scratch_iters = run_chomp(use_warmstart=False)
refine_free,  t_refine_only, refine_iters = run_chomp(use_warmstart=True)
t_refine = t_model + t_refine_only

summary = pd.DataFrame({
    "collision_free": [f"{model_free}/{N_COMPARE}", f"{scratch_free}/{N_COMPARE}", f"{refine_free}/{N_COMPARE}"],
    "free_rate":      [model_free/N_COMPARE, scratch_free/N_COMPARE, refine_free/N_COMPARE],
    "mean_iters":     [0, round(scratch_iters, 0), round(refine_iters, 0)],
    "ms_per_sample":  [round(t_model*1e3, 2), round(t_scratch*1e3, 1), round(t_refine*1e3, 1)],
}, index=["model", "CHOMP from scratch", "model + CHOMP"])
summary